In [1]:
# !pip install google-play-scraper

In [2]:
""" The Scraper Script
Use the google_play_scraper library. 
You need the App IDs for the three banks (usually found in the Play Store URL).
•	CBE: com.combanketh.mobilebanking
•	BOA: com.boa.boaMobileBanking
•	Dashen: com.dashen.dashensuperapp """

' The Scraper Script\nUse the google_play_scraper library. \nYou need the App IDs for the three banks (usually found in the Play Store URL).\n•\tCBE: com.combanketh.mobilebanking\n•\tBOA: com.boa.boaMobileBanking\n•\tDashen: com.dashen.dashensuperapp '

In [3]:
import pandas as pd
from google_play_scraper import Sort, reviews
print("Library loaded successfully.")

def scrape_bank_reviews(app_id, bank_name):
    result, _ = reviews(
        app_id,
        lang='en', 
        country='et', 
        sort=Sort.NEWEST, 
        count=500 # Aim for 500 to ensure 400 clean ones
    )
    df = pd.DataFrame(result)
    df['bank'] = bank_name
    return df

if __name__ == "__main__":
    # Corrected Google Play App IDs
    # Note: If 'com.cbe.cbe_mobile' doesn't fetch enough, use 'com.cbe.cbebirr' depending on which app you want to target
    apps_to_scrape = {

        'CBE': 'com.combanketh.mobilebanking',
        'BOA': 'com.boa.boaMobileBanking',
        'Dashen': 'com.dashen.dashensuperapp'
    }
    
    scraped_dfs = []
    
    for bank_name, app_id in apps_to_scrape.items():
        print(f"Scraping {bank_name} using ID: {app_id}...")
        df_bank = scrape_bank_reviews(app_id, bank_name)
        
        print(f"Found {len(df_bank)} reviews for {bank_name}.")
        if not df_bank.empty:
            scraped_dfs.append(df_bank)
        else:
            print(f"⚠️ Warning: No data returned for {bank_name}. Check the App ID or country code.")

    # Only combine and save if we actually got data
    if scraped_dfs:
        all_reviews = pd.concat(scraped_dfs, ignore_index=True)
        all_reviews.to_csv('../data/raw/bank_reviews.csv', index=False)
        print("\n✅ Success! Scraping completed and saved to bank_reviews.csv")
        print(f"Total reviews collected: {len(all_reviews)}")
        print(all_reviews['bank'].value_counts()) # Shows breakdown per bank
    else:
        print("\n❌ Error: No reviews were collected for any bank. CSV not updated.")

    result, _ = reviews(
        app_id,
        lang='en', 
        country='et',  # Changed from 'us' to 'et' (Ethiopia)
        sort=Sort.NEWEST, 
        count=500
    )

Library loaded successfully.
Scraping CBE using ID: com.combanketh.mobilebanking...
Found 500 reviews for CBE.
Scraping BOA using ID: com.boa.boaMobileBanking...
Found 500 reviews for BOA.
Scraping Dashen using ID: com.dashen.dashensuperapp...
Found 500 reviews for Dashen.

✅ Success! Scraping completed and saved to bank_reviews.csv
Total reviews collected: 1500
bank
CBE       500
BOA       500
Dashen    500
Name: count, dtype: int64


C:\Users\Almazt\AppData\Local\Temp\ipykernel_22480\905968020.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_reviews = pd.concat(scraped_dfs, ignore_index=True)


""" Preprocessing and Cleaning Steps
After scraping, we need to clean the data to ensure we have 400 clean reviews for each bank. This involves:
1. Removing any duplicate reviews
2. Filtering out reviews that are too short or contain inappropriate content
3. Standardizing the format of the reviews
4. Handling missing values appropriately """
# Example of cleaning the data
def clean_reviews(df):
    # Remove duplicates
    df = df.drop_duplicates(subset=['content'])
    
    # Filter out short reviews (e.g., less than 10 characters)
    df = df[df['content'].str.len() >= 10]
    
    # Optionally, filter out reviews with inappropriate content using a simple keyword filter
    inappropriate_keywords = ['bad', 'worst', 'terrible']  # Example keywords
    df = df[~df['content'].str.contains('|'.join(inappropriate_keywords), case=False)]
    
    # Handle missing values (e.g., fill with 'No review' or drop)
    df['content'] = df['content'].fillna('No review')
    
    return df
# Apply cleaning to each bank's reviews
if scraped_dfs:
    cleaned_dfs = [clean_reviews(df) for df in scraped_dfs]
    all_cleaned_reviews = pd.concat(cleaned_dfs, ignore_index=True)
    all_cleaned_reviews.to_csv('cleaned_bank_reviews.csv', index=False)
    print("\n✅ Cleaning completed and saved to cleaned_bank_reviews.csv")
    print(f"Total clean reviews collected: {len(all_cleaned_reviews)}")
else:
    print("\n❌ Error: No reviews to clean. CSV not updated.")
    

In [4]:
""" Preprocessing and Cleaning Steps
After scraping, we need to clean the data to ensure we have 400 clean reviews for each bank. This involves:
1. Removing any duplicate reviews
2. Filtering out reviews that are too short or contain inappropriate content
3. Standardizing the format of the reviews
4. Handling missing values appropriately """
# Example of cleaning the data
def clean_reviews(df):
    # Remove duplicates
    df = df.drop_duplicates(subset=['content'])
    
    # Filter out short reviews (e.g., less than 10 characters)
    df = df[df['content'].str.len() >= 10]
    
    # Optionally, filter out reviews with inappropriate content using a simple keyword filter
    inappropriate_keywords = ['bad', 'worst', 'terrible']  # Example keywords
    df = df[~df['content'].str.contains('|'.join(inappropriate_keywords), case=False)]
    
    # Handle missing values (e.g., fill with 'No review' or drop)
    df['content'] = df['content'].fillna('No review')
    
    return df
# Apply cleaning to each bank's reviews
if scraped_dfs:
    cleaned_dfs = [clean_reviews(df) for df in scraped_dfs]
    all_cleaned_reviews = pd.concat(cleaned_dfs, ignore_index=True)
    all_cleaned_reviews.to_csv('../data/cleaned/cleaned_bank_reviews.csv', index=False)
    print("\n✅ Cleaning completed and saved to cleaned_bank_reviews.csv")
    print(f"Total clean reviews collected: {len(all_cleaned_reviews)}")
    print(all_cleaned_reviews['bank'].value_counts()) # Shows breakdown per bank
else:
    print("\n❌ Error: No reviews to clean. CSV not updated.")
    


✅ Cleaning completed and saved to cleaned_bank_reviews.csv
Total clean reviews collected: 862
bank
Dashen    318
BOA       278
CBE       266
Name: count, dtype: int64


C:\Users\Almazt\AppData\Local\Temp\ipykernel_22480\3353281957.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cleaned_reviews = pd.concat(cleaned_dfs, ignore_index=True)
